# Expected Value vs. Most Likely Outcome

**Monte Carlo Simulation Study Note**

This notebook reproduces every calculation and visualization from my study note:

> **“Why Expected Value Is Often the Right Decision Tool, Even When the Most Likely Outcome Points Elsewhere”**

It demonstrates why the **mode** (most likely outcome) can be misleading and why **expected value (EV)** is the superior decision metric under uncertainty.

---

**I am aiming to evolve this work.**

This notebook represents **Part 1** — the foundational intuition and basic Monte Carlo implementation.

**Part 2 (in progress)** — *Expected Value in Practice: Monte Carlo Simulation for Real A/B Testing & Experimentation* — will extend the same concepts to realistic experimentation scenarios, including:
- Continuous outcomes and revenue lift
- Multiple experiment variants
- Bayesian expected value (Beta-Binomial priors)
- Statistical power & sample-size planning
- Sensitivity analysis and advanced risk metrics

The evolution will be published in the same GitHub series and linked from this notebook once complete.

---

In [1]:
# Import required libraries
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path so we can import our reusable modules
sys.path.append('..')

from src.simulation import run_monte_carlo, STRATEGIES
from src.visualization import plot_outcome_distributions, plot_ev_convergence

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ All libraries loaded successfully")

✅ All libraries loaded successfully


## 1. The Two Example Pairs from the Study Note

In [2]:
# Display the strategies
print("Strategies defined in src/simulation.py:")
for name, strategy in STRATEGIES.items():
    print(f"{name}: {strategy}")

Strategies defined in src/simulation.py:
Option A: [(0.9, 10), (0.1, -5)]
Option B: [(0.4, 30), (0.6, 0)]
Strategy 1: [(0.8, 5), (0.2, -20)]
Strategy 2: [(0.45, 15), (0.55, 0)]


## 2. Run Monte Carlo Simulations (10,000 trials each)

In [3]:
results = {}

for name, strategy in STRATEGIES.items():
    print(f"Running simulation for {name}...")
    results[name] = run_monte_carlo(strategy, n_trials=10000)

print("\n✅ All simulations completed")

Running simulation for Option A...
Running simulation for Option B...
Running simulation for Strategy 1...
Running simulation for Strategy 2...

✅ All simulations completed


## 3. Results Summary Table

In [4]:
# Create comparison table
data = []
for name, res in results.items():
    data.append({
        "Strategy": name,
        "Mode": res["mode"],
        "EV (analytic)": round(res["ev_analytic"], 2),
        "EV (simulated)": round(res["ev_simulated"], 2),
        "Std Dev": round(res["std"], 2),
        "5% VaR": round(np.percentile(res["trials"], 5), 1)
    })

df = pd.DataFrame(data)
display(df.style.format({
    "EV (analytic)": "{:.2f}",
    "EV (simulated)": "{:.2f}",
    "Std Dev": "{:.2f}"
}).set_caption("Comparison of Analytic vs. Simulated Results (10,000 trials)"))

,Strategy,Mode,EV (analytic),EV (simulated),Std Dev,5% VaR
0,Option A,10.000000,8.50,8.56,4.42,-5.000000
1,Option B,0.000000,12.00,12.17,14.73,0.000000
2,Strategy 1,5.000000,0.00,0.10,9.93,-20.000000
3,Strategy 2,0.000000,6.75,6.87,7.47,0.000000


## 4. Visualization 1: Outcome Distributions

In [5]:
plot_outcome_distributions(results, save_path="../results/ev_histogram.png")
plt.show()

✅ Histogram saved to ../results/ev_histogram.png


## 5. Visualization 2: Law of Large Numbers Convergence

In [6]:
plot_ev_convergence(results, save_path="../results/ev_convergence.png")
plt.show()

✅ Convergence plot saved to ../results/ev_convergence.png


## 6. Key Takeaways

- The **most likely outcome (mode)** and **expected value** frequently disagree when payoffs are asymmetric.
- In both example pairs, the option that “feels safer” (higher mode) actually destroys value.
- Over repeated decisions, the higher-EV strategy reliably wins — as proven by the convergence plots (Law of Large Numbers).
- This is exactly why analysts combine analytical EV with Monte Carlo simulation when evaluating product features, marketing campaigns, or experimentation roadmaps.
